### Check GPU availability and display GPU information

In [1]:
gpu_info = !nvidia-smi
gpu_info = "\n".join(gpu_info)
if gpu_info.find("failed") >= 0:
    print("Not connected to a GPU")
else:
    print(gpu_info)


Sun Mar  8 05:54:11 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   47C    P8             13W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

### Install required Python packages (datasets, transformers, torchaudio, jiwer, accelerate, evaluate)

In [2]:
%%capture
!pip install datasets==3.6.0
!pip install transformers==4.57.1
!pip install torchaudio
!pip install jiwer
!pip install accelerate -U
!pip install evaluate


### Load the Dhivehi (Maldivian) Common Voice dataset (train+validation and test splits)

In [3]:
from datasets import load_dataset

common_voice_train = load_dataset("fsicoli/common_voice_22_0", "dv", split="train+validation", trust_remote_code=True)
common_voice_test = load_dataset("fsicoli/common_voice_22_0", "dv", split="test", trust_remote_code=True)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

common_voice_22_0.py: 0.00B [00:00, ?B/s]

languages.py: 0.00B [00:00, ?B/s]

release_stats.py: 0.00B [00:00, ?B/s]

...


Reading metadata...: 2222it [00:00, 116834.15it/s]


Generating other split: 0 examples [00:00, ? examples/s]


Reading metadata...: 0it [00:00, ?it/s]
Reading metadata...: 15104it [00:00, 105691.01it/s]


Generating invalidated split: 0 examples [00:00, ? examples/s]


Reading metadata...: 1652it [00:00, 106950.32it/s]


### Remove unused metadata columns (accent, age, client_id, etc.) from train and test sets and display random samples

In [4]:
common_voice_train = common_voice_train.remove_columns(
    ["accent", "age", "client_id", "down_votes", "gender", "locale", "segment", "up_votes"]
)
common_voice_test = common_voice_test.remove_columns(
    ["accent", "age", "client_id", "down_votes", "gender", "locale", "segment", "up_votes"]
)

from datasets import ClassLabel
import random
import pandas as pd
from IPython.display import display, HTML

def show_random_elements(dataset, num_examples=10):
    assert num_examples <= len(dataset), "Can't pick more elements than there are in the dataset."
    picks = []
    for _ in range(num_examples):
        pick = random.randint(0, len(dataset) - 1)
        while pick in picks:
            pick = random.randint(0, len(dataset) - 1)
        picks.append(pick)
    df = pd.DataFrame(dataset[picks])
    display(HTML(df.to_html()))

# Select the 'train' split before calling show_random_elements
show_random_elements(common_voice_train.remove_columns(["path", "audio"]), num_examples=10)


,sentence,variant
0,އެލީޝާއަށް ގޮތެއް ވެއްޖިއްޔާ ނައުފަލު ވަރަށް ދެރަވާނެ,
1,އޭނާއަށްޓަކައި އަލިކަމާއި އުފާވެރިކަމުގެ މަޞްދަރަކީ އަހަރެން,
2,ދޮންތަ ފެށި ޖުމުލަ ހުއްޓުވީ އުޒައިރުބެ ގެ ހުޝްޝް އިން,
3,މަސްގޮނޑި ރީތިކޮށް ހިއްކާލުމަށްފަހު ރާހުލް ފެންވަރާލައިގެން ކޮޓަރިއަށް ދިޔައިރު އެ މޫނުމަތީގައި ވަނީ ލާނެތް ހިނިތުންވުމެއް,
4,ޝަކީބުގެ މިޖަވާބަކީ އަހަރެން ބޭނުންވި ޖަވާބެއް ނޫން,
5,އެއީ މިނިސްޓްރީ އޮފް އިސްލާމިކް އެފެއަރޒުން ހިންގަވާ މަޝްރޫޢެއްގެ ދަށުން ފާރޫޤާ ހަވާލު ކުރެއްވުނު މަސައްކަތެއް,
6,ގިނަ އަންނައުނު ހުރީ ދާވެފަ,
7,މަހާ ކޮޓަރީގެ ދޮރުހުޅުވާލުމާއެކު އޭނާ ސިހިގެން ދިޔައީ ދޮރު ކުރިމަތީގައި ޢަލީބެ ހުރުމުން,
8,ބުރުގާ އެޅުމަކީ ބާލިޣުވެއްޖެ ކޮންމެ މުސްލިމް އަންހެނެއްގެ މައްޗަށް މާތްﷲ ވާޖިބު ކުރައްވާފައިވާ ކަމެއްކަން ޝިފްޒާމެންފަދަ މީހުންނަށް ނޭނގެނީކަންނޭނގެ,
9,ހިތާއި ސިކުނޑިޔަށް ފިނިކަންދޭނޭ ބަނގު އެއްޗެހި ކައި ނުވަތަ ބޮއި ހަދައިގެން,


### Define and apply text cleaning: remove punctuation, special characters, and filter out Arabic script rows

In [5]:
import re

# remove punctuation etc. (expanded to include – and ’)
chars_to_remove_regex = r"""[\,\?\.\!\-\;\:\"\“\%\‘\”\�\'\»\«\–\’]"""

def remove_special_characters(batch):
    batch["sentence"] = re.sub(chars_to_remove_regex, "", batch["sentence"]).lower().strip()
    return batch

common_voice_train = common_voice_train.map(remove_special_characters)
common_voice_test  = common_voice_test.map(remove_special_characters)

# remove rows that contain Arabic characters (expanded ranges)
# - Arabic: \u0600-\u06FF
# - Arabic Supplement/Extended and presentation forms commonly used for ligatures:
#   \u0750-\u077F, \u08A0-\u08FF, \uFB50-\uFDFF, \uFE70-\uFEFF
arabic_pattern = re.compile(r"[\u0600-\u06FF\u0750-\u077F\u08A0-\u08FF\uFB50-\uFDFF\uFE70-\uFEFF]")

def remove_arabic_rows(batch):
    # True if NO Arabic chars → keep row
    return not bool(arabic_pattern.search(batch["sentence"]))

common_voice_train = common_voice_train.filter(remove_arabic_rows)
common_voice_test  = common_voice_test.filter(remove_arabic_rows)

Map:   0%|          | 0/4897 [00:00<?, ? examples/s]

Map:   0%|          | 0/2222 [00:00<?, ? examples/s]

Filter:   0%|          | 0/4897 [00:00<?, ? examples/s]

Filter:   0%|          | 0/2222 [00:00<?, ? examples/s]

### Extract the unique character vocabulary from train and test sets

In [6]:
def extract_all_chars(batch):
    all_text = " ".join(batch["sentence"])
    vocab = list(set(all_text))
    return {"vocab": [vocab], "all_text": [all_text]}

vocab_train = common_voice_train.map(
    extract_all_chars,
    batched=True,
    batch_size=-1,
    keep_in_memory=True,
    remove_columns=common_voice_train.column_names,
)
vocab_test = common_voice_test.map(
    extract_all_chars,
    batched=True,
    batch_size=-1,
    keep_in_memory=True,
    remove_columns=common_voice_test.column_names,
)

vocab_list = list(set(vocab_train["vocab"][0]) | set(vocab_test["vocab"][0]))
vocab_dict = {v: k for k, v in enumerate(sorted(vocab_list))}
print(vocab_dict)


Map:   0%|          | 0/4600 [00:00<?, ? examples/s]

Map:   0%|          | 0/2077 [00:00<?, ? examples/s]

{' ': 0, 'ހ': 1, 'ށ': 2, 'ނ': 3, 'ރ': 4, 'ބ': 5, 'ޅ': 6, 'ކ': 7, 'އ': 8, 'ވ': 9, 'މ': 10, 'ފ': 11, 'ދ': 12, 'ތ': 13, 'ލ': 14, 'ގ': 15, 'ޏ': 16, 'ސ': 17, 'ޑ': 18, 'ޒ': 19, 'ޓ': 20, 'ޔ': 21, 'ޕ': 22, 'ޖ': 23, 'ޗ': 24, 'ޘ': 25, 'ޙ': 26, 'ޚ': 27, 'ޛ': 28, 'ޝ': 29, 'ޞ': 30, 'ޟ': 31, 'ޠ': 32, 'ޡ': 33, 'ޢ': 34, 'ޣ': 35, 'ޤ': 36, 'ޥ': 37, 'ަ': 38, 'ާ': 39, 'ި': 40, 'ީ': 41, 'ު': 42, 'ޫ': 43, 'ެ': 44, 'ޭ': 45, 'ޮ': 46, 'ޯ': 47, 'ް': 48}


### Remove any remaining Latin characters from transcriptions and re-extract the character vocabulary

In [7]:
def remove_latin_characters(batch):
    batch["sentence"] = re.sub(r"[a-z]+", "", batch["sentence"])
    return batch

# remove latin characters
common_voice_train = common_voice_train.map(remove_latin_characters)
common_voice_test = common_voice_test.map(remove_latin_characters)

# extract unique characters again
vocab_train = common_voice_train.map(
    extract_all_chars,
    batched=True,
    batch_size=-1,
    keep_in_memory=True,
    remove_columns=common_voice_train.column_names,
)
vocab_test = common_voice_test.map(
    extract_all_chars,
    batched=True,
    batch_size=-1,
    keep_in_memory=True,
    remove_columns=common_voice_test.column_names,
)

vocab_list = list(set(vocab_train["vocab"][0]) | set(vocab_test["vocab"][0]))
vocab_dict = {v: k for k, v in enumerate(sorted(vocab_list))}
print(vocab_dict)


Map:   0%|          | 0/4600 [00:00<?, ? examples/s]

Map:   0%|          | 0/2077 [00:00<?, ? examples/s]

Map:   0%|          | 0/4600 [00:00<?, ? examples/s]

Map:   0%|          | 0/2077 [00:00<?, ? examples/s]

{' ': 0, 'ހ': 1, 'ށ': 2, 'ނ': 3, 'ރ': 4, 'ބ': 5, 'ޅ': 6, 'ކ': 7, 'އ': 8, 'ވ': 9, 'މ': 10, 'ފ': 11, 'ދ': 12, 'ތ': 13, 'ލ': 14, 'ގ': 15, 'ޏ': 16, 'ސ': 17, 'ޑ': 18, 'ޒ': 19, 'ޓ': 20, 'ޔ': 21, 'ޕ': 22, 'ޖ': 23, 'ޗ': 24, 'ޘ': 25, 'ޙ': 26, 'ޚ': 27, 'ޛ': 28, 'ޝ': 29, 'ޞ': 30, 'ޟ': 31, 'ޠ': 32, 'ޡ': 33, 'ޢ': 34, 'ޣ': 35, 'ޤ': 36, 'ޥ': 37, 'ަ': 38, 'ާ': 39, 'ި': 40, 'ީ': 41, 'ު': 42, 'ޫ': 43, 'ެ': 44, 'ޭ': 45, 'ޮ': 46, 'ޯ': 47, 'ް': 48}


### Finalize the vocabulary: replace space with pipe delimiter, add [UNK] and [PAD] tokens, and save vocab.json

In [8]:
# use "|" as word delimiter instead of space
vocab_dict["|"] = vocab_dict[" "]
del vocab_dict[" "]

# add special tokens
vocab_dict["[UNK]"] = len(vocab_dict)
vocab_dict["[PAD]"] = len(vocab_dict)

print("Vocab size:", len(vocab_dict))

import json
with open("vocab.json", "w", encoding="utf-8") as vocab_file:
    json.dump(vocab_dict, vocab_file, ensure_ascii=False)


Vocab size: 51


### Initialize the CTC tokenizer, SeamlessM4T feature extractor, and Wav2Vec2-BERT processor

In [9]:
from transformers import Wav2Vec2CTCTokenizer
from transformers import SeamlessM4TFeatureExtractor
from transformers import Wav2Vec2BertProcessor

tokenizer = Wav2Vec2CTCTokenizer.from_pretrained(
    "./",
    unk_token="[UNK]",
    pad_token="[PAD]",
    word_delimiter_token="|",
)

feature_extractor = SeamlessM4TFeatureExtractor(
    feature_size=80,
    num_mel_bins=80,
    sampling_rate=16000,
    padding_value=0.0,
)

processor = Wav2Vec2BertProcessor(
    feature_extractor=feature_extractor,
    tokenizer=tokenizer,
)


### Resample all audio to 16kHz and preview a random training sample

In [10]:
from datasets import Audio

common_voice_train = common_voice_train.cast_column("audio", Audio(sampling_rate=16_000))
common_voice_test = common_voice_test.cast_column("audio", Audio(sampling_rate=16_000))

print(common_voice_train[0]["audio"])

import IPython.display as ipd
import numpy as np
import random

rand_int = random.randint(0, len(common_voice_train) - 1)
print("Target text:", common_voice_train[rand_int]["sentence"])
print("Input array shape:", common_voice_train[rand_int]["audio"]["array"].shape)
print("Sampling rate:", common_voice_train[rand_int]["audio"]["sampling_rate"])

ipd.Audio(
    data=common_voice_train[rand_int]["audio"]["array"],
    autoplay=True,
    rate=16000,
)


{'path': '/root/.cache/huggingface/datasets/downloads/extracted/e24cb6e0896730e3c68f4266eadc6dbf6150fbe071ba9968bf13481404180b10/dv_train_0/common_voice_dv_18580646.mp3', 'array': array([1.45519152e-11, 1.01863407e-10, 1.30967237e-10, ...,
       7.25846985e-06, 1.29183172e-06, 5.80571304e-06]), 'sampling_rate': 16000}
Target text: ބާރކޯޑު ޕްރިންޓްކޮށްދޭނެ ފަރާތެއް ހޯދުމަށް
Input array shape: (69504,)
Sampling rate: 16000


### Define the dataset preparation function: extract audio features and encode text labels, then apply to train and test sets

In [11]:
def prepare_dataset(batch):
    audio = batch["audio"]
    batch["input_features"] = processor(
        audio["array"],
        sampling_rate=audio["sampling_rate"],
    ).input_features[0]
    batch["input_length"] = len(batch["input_features"])
    batch["labels"] = processor(text=batch["sentence"]).input_ids
    return batch

common_voice_train = common_voice_train.map(
    prepare_dataset,
    remove_columns=common_voice_train.column_names,
)
common_voice_test = common_voice_test.map(
    prepare_dataset,
    remove_columns=common_voice_test.column_names,
)


Map:   0%|          | 0/4600 [00:00<?, ? examples/s]

Map:   0%|          | 0/2077 [00:00<?, ? examples/s]

### Define a custom data collator that pads input features and labels for CTC training

In [12]:
import torch
from dataclasses import dataclass
from typing import Any, Dict, List, Optional, Union
import evaluate

from transformers import Wav2Vec2BertProcessor

@dataclass
class DataCollatorCTCWithPadding:
    processor: Wav2Vec2BertProcessor
    padding: Union[bool, str] = True

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        # split inputs and labels since they have to be of different lengths
        input_features = [{"input_features": f["input_features"]} for f in features]
        label_features = [{"input_ids": f["labels"]} for f in features]

        batch = self.processor.pad(
            input_features,
            padding=self.padding,
            return_tensors="pt",
        )
        labels_batch = self.processor.pad(
            labels=label_features,
            padding=self.padding,
            return_tensors="pt",
        )

        # replace padding with -100 to ignore loss correctly
        labels = labels_batch["input_ids"].masked_fill(labels_batch["attention_mask"].ne(1), -100)
        batch["labels"] = labels
        return batch

data_collator = DataCollatorCTCWithPadding(processor=processor, padding=True)


### Define the evaluation metrics function computing WER and CER using greedy decoding

In [13]:
import numpy as np

wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")

def compute_metrics(pred):
    pred_logits = pred.predictions
    label_ids = pred.label_ids

    pred_ids = np.argmax(pred_logits, axis=-1)
    pred_str = processor.batch_decode(pred_ids)

    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id
    label_str = processor.batch_decode(label_ids, group_tokens=False)

    wer = wer_metric.compute(predictions=pred_str, references=label_str)
    cer = cer_metric.compute(predictions=pred_str, references=label_str)

    return {"wer": wer, "cer": cer}

### Load the pre-trained W2V-BERT 2.0 model with an adapter and CTC head for fine-tuning

In [14]:
from transformers import Wav2Vec2BertForCTC

model = Wav2Vec2BertForCTC.from_pretrained(
    "facebook/w2v-bert-2.0",
    add_adapter=True,
    pad_token_id=processor.tokenizer.pad_token_id,
    vocab_size=len(processor.tokenizer),
)


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/2.32G [00:00<?, ?B/s]

Some weights of Wav2Vec2BertForCTC were not initialized from the model checkpoint at facebook/w2v-bert-2.0 and are newly initialized: ['lm_head.bias', 'lm_head.weight', 'wav2vec2_bert.adapter.layers.0.ffn.intermediate_dense.bias', 'wav2vec2_bert.adapter.layers.0.ffn.intermediate_dense.weight', 'wav2vec2_bert.adapter.layers.0.ffn.output_dense.bias', 'wav2vec2_bert.adapter.layers.0.ffn.output_dense.weight', 'wav2vec2_bert.adapter.layers.0.ffn_layer_norm.bias', 'wav2vec2_bert.adapter.layers.0.ffn_layer_norm.weight', 'wav2vec2_bert.adapter.layers.0.residual_conv.bias', 'wav2vec2_bert.adapter.layers.0.residual_conv.weight', 'wav2vec2_bert.adapter.layers.0.residual_layer_norm.bias', 'wav2vec2_bert.adapter.layers.0.residual_layer_norm.weight', 'wav2vec2_bert.adapter.layers.0.self_attn.linear_k.bias', 'wav2vec2_bert.adapter.layers.0.self_attn.linear_k.weight', 'wav2vec2_bert.adapter.layers.0.self_attn.linear_out.bias', 'wav2vec2_bert.adapter.layers.0.self_attn.linear_out.weight', 'wav2vec2_ber

### Configure training hyperparameters: batch size, learning rate, evaluation strategy, checkpointing, and early stopping

In [15]:
from transformers import TrainingArguments, Trainer, Wav2Vec2BertForCTC, EarlyStoppingCallback

training_args = TrainingArguments(
    output_dir="./outputs",
    group_by_length=True,
    per_device_train_batch_size=16,
    gradient_accumulation_steps=2,
    num_train_epochs=10,
    gradient_checkpointing=True,
    fp16=True,

    eval_strategy="steps",     # FIXED
    eval_steps=300,

    logging_strategy="steps",
    logging_steps=300,

    save_strategy="steps",
    save_steps=2700,

    learning_rate=5e-5,
    warmup_steps=500,
    save_total_limit=2,

    load_best_model_at_end=True,
    metric_for_best_model="eval_wer",   # FIXED
    greater_is_better=False,

    push_to_hub=False,
    report_to="none",
)

trainer = Trainer(
    model=model,
    data_collator=data_collator,
    args=training_args,
    compute_metrics=compute_metrics,
    train_dataset=common_voice_train,
    eval_dataset=common_voice_test,
    tokenizer=processor
)


/tmp/ipykernel_4668/723348825.py:33: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


### Start the fine-tuning training run

In [16]:
trainer.train()


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 52, 'bos_token_id': 51}.


Step,Training Loss,Validation Loss,Wer,Cer
300,2997.030400,264.059265,0.625148,0.115825


Step,Training Loss,Validation Loss,Wer,Cer
300,2997.030400,264.059265,0.625148,0.115825
600,831.652200,176.006287,0.539860,0.086260
900,644.286000,144.783463,0.456057,0.069944
1200,537.438800,133.436966,0.428518,0.064874


TrainOutput(global_step=1440, training_loss=1117.8111979166667, metrics={'train_runtime': 5018.1487, 'train_samples_per_second': 9.167, 'train_steps_per_second': 0.287, 'total_flos': 7.110251700446906e+18, 'train_loss': 1117.8111979166667, 'epoch': 10.0})

### Run greedy CTC inference on the test set and compute WER/CER

In [18]:
from transformers import AutoModelForCTC, Wav2Vec2BertProcessor
import torch
import numpy as np


all_pred = []
all_refs = []

checkpoint_dir = "./outputs/checkpoint-1440"

model = AutoModelForCTC.from_pretrained(checkpoint_dir).to("cuda")
processor = Wav2Vec2BertProcessor.from_pretrained(checkpoint_dir)

for ex in common_voice_test:
    input_feats = torch.tensor(ex["input_features"]).unsqueeze(0).to(model.device)

    with torch.no_grad():
        logits = model(input_feats).logits

    pred_ids = torch.argmax(logits, dim=-1)
    pred_text = processor.batch_decode(pred_ids)[0]
    all_pred.append(pred_text)

    ref_text = processor.decode(ex["labels"], group_tokens=False).lower()
    all_refs.append(ref_text)

wer = wer_metric.compute(predictions=all_pred, references=all_refs)
cer = cer_metric.compute(predictions=all_pred, references=all_refs)

print(f"WER (ASR only, greedy): {wer:.4f}")
print(f"CER (ASR only, greedy): {cer:.4f}")

for idx in range(3):
    print("\n--- Example", idx, "---")
    print("REF :", all_refs[idx])
    print("PRED:", all_pred[idx])

WER (ASR only, greedy): 0.4127
CER (ASR only, greedy): 0.0619

--- Example 0 ---
REF : ޕިކަޕް މަރާމާތުކޮށްދޭނެ ފަރާތެއް ހޯދުން
PRED: ޕިކަޕް މަރާމާތުކޮށްދޭނެ ފަރާތެއް ހޯދުން

--- Example 1 ---
REF : އަންޖޫދާ މަޑުމަޑުން އެހެން ބުނެލުމުން ރިޝްކާ ވެސް އިތުރު ސުވާލެއް ނުކުރޭ
PRED: އަންޖޫދާ މަޑުމަޑުން އެހެން ބުނެލުމުން ރިޝްކާވެސް އިތުރު ސުވާލެއް ނުކުރޭ

--- Example 2 ---
REF : އެސްފިޔަތަކުން ވަނީ ނިދި ގެއްލިފަ
PRED: އެސްފިޔަތަކުން ވަނީ ނިދިގެއްލިފަ
